```markdown
# 5. 모델 품질 모니터링 (Model Quality Monitoring)

이 섹션에서는 서비스 중인 모델의 입력 데이터 분포 변화(Data Drift)를 감지하는 프로세스를 실습합니다.

*   **Base Data**: 모델 학습 시 사용했거나 과거 정상 상태의 데이터 기준점
*   **Current Data**: 현재 서비스에서 수집되고 있는 실시간 데이터

<img width="180" alt="ops" src="https://www.gstatic.com/images/branding/googlelogo/2x/googlelogo_color_92x30dp.png" />
```

In [ ]:
from google.cloud import bigquery
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

import uuid
import datetime

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
PROJECT_ID = "gde-project-aicloud"                  # @param { "type": "string" }
DATASET_ID = "example"                         # @param { "type": "string" }
TABLE_ID = "data_drift"                        # @param { "type": "string" }

In [ ]:
from google.cloud import bigquery
import google.auth

# Get credentials and project ID explicitly from the Colab environment
credentials, default_project = google.auth.default()

try:
    active_project = PROJECT_ID
except NameError:
    active_project = default_project or "gde-project-aicloud"
    PROJECT_ID = active_project

# Pass credentials explicitly to the client to avoid RefreshError
client = bigquery.Client(project=active_project, credentials=credentials)
print(f"BigQuery client initialized for project: {active_project}")

BigQuery client initialized for project: gde-project-aicloud


In [ ]:
import google.auth
from google.cloud import bigquery
from google.colab import auth
from google.api_core import exceptions

# Re-authenticate to ensure fresh tokens
auth.authenticate_user()
credentials, detected_project_id = google.auth.default()

# Fallback for PROJECT_ID
PROJECT_ID = detected_project_id or "gde-project-aicloud"
DATASET_ID = "example_drift_dataset"
TABLE_ID = "data_drift"

client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

# 1. Ensure Dataset exists
dataset_exists = False
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
try:
    client.get_dataset(dataset_ref)
    print(f"Dataset {DATASET_ID} exists in {PROJECT_ID}.")
    dataset_exists = True
except exceptions.NotFound:
    try:
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        client.create_dataset(dataset, timeout=30)
        print(f"Created dataset {DATASET_ID}")
        dataset_exists = True
    except exceptions.Forbidden:
        print(f"Permission denied to create dataset in {PROJECT_ID}. Please ensure you have sufficient IAM permissions or use a project where you are an owner.")

# 2. Ensure Table exists only if dataset exists
if dataset_exists:
    full_table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
    try:
        client.get_table(full_table_id)
        print(f"Table {full_table_id} exists.")
    except exceptions.NotFound:
        schema = [
            bigquery.SchemaField("movie_id", "STRING"),
            bigquery.SchemaField("watch_time", "INTEGER"),
            bigquery.SchemaField("date", "INTEGER"),
        ]
        table = bigquery.Table(full_table_id, schema=schema)
        client.create_table(table)
        print(f"Created table {full_table_id}")

Permission denied to create dataset in gde-project-aicloud. Please ensure you have sufficient IAM permissions or use a project where you are an owner.


In [ ]:
def insert_new_line(watch_time: int, current_datetime: datetime.datetime) -> None:
    rows_to_insert = [
        {
            "movie_id": str(uuid.uuid4()),
            "watch_time": int(watch_time),
            "date": int(current_datetime.timestamp()),
        },
    ]

    errors = client.insert_rows_json(full_table_id, rows_to_insert)
    if errors != []:
        print("Encountered errors while inserting rows: {}".format(errors))

In [ ]:
import datetime
import uuid
import numpy as np
from tqdm import tqdm
import google.auth
from google.cloud import bigquery

# Ensure credentials and project are determined correctly
credentials, detected_project = google.auth.default()
target_project = (globals().get('PROJECT_ID')) or detected_project or "gde-project-aicloud"
target_dataset = (globals().get('DATASET_ID')) or "example_drift_dataset"
target_table = (globals().get('TABLE_ID')) or "data_drift"

full_table_id = f"{target_project}.{target_dataset}.{target_table}"
client = bigquery.Client(project=target_project, credentials=credentials)

# Only proceed if we detected the dataset exists in the previous step
if globals().get('dataset_exists', False):
    total_iterations = 10000
    current_datetime = datetime.datetime.now(datetime.timezone.utc)
    row_data = []

    print(f"Generating data for project: {target_project}...")
    for i in tqdm(range(total_iterations)):
        if i >= total_iterations // 2:
            mean, std_dev = 4800, 1200
            watch_time = np.clip(np.random.normal(mean, std_dev), 2400, 7200)
        else:
            mean, std_dev = 2500, 1250
            watch_time = np.clip(np.random.normal(mean, std_dev), 0, 5000)

        row_data.append({
            "movie_id": str(uuid.uuid4()),
            "watch_time": int(watch_time),
            "date": int(current_datetime.timestamp()),
        })
        current_datetime += datetime.timedelta(seconds=np.random.uniform(1.0, 120.0))

    print(f"Inserting {len(row_data)} rows into {full_table_id}...")
    try:
        errors = client.insert_rows_json(full_table_id, row_data)
        if not errors:
            print("Successfully inserted all rows.")
        else:
            print(f"Encountered errors: {errors}")
    except Exception as e:
        print(f"Failed to insert rows: {e}")
else:
    print("Dataset does not exist or access was denied. Skipping data insertion.")

Dataset does not exist or access was denied. Skipping data insertion.


In [ ]:
!pip install facets-overview

In [ ]:
query = (
    f"""
    SELECT watch_time
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    ORDER BY date DESC
    """
)
df = client.query(query).to_dataframe(create_bqstorage_client=False)
df.head()

Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/gde-project-aicloud/jobs?prettyPrint=false: Access Denied: Project gde-project-aicloud: User does not have bigquery.jobs.create permission in project gde-project-aicloud.

Location: None
Job ID: dcd36118-3e42-411c-aab7-3a8320d6dc02


In [ ]:
df_len = df.shape[0]
df["watch_time"] = df.watch_time.astype(float)
df_current = df.iloc[:df_len//2]
df_base = df.iloc[df_len//2:]

In [ ]:
import matplotlib.pyplot as plt

# 데이터프레임이 준비되었는지 확인 후 시각화
if 'df_current' in locals() and not df_current.empty:
    plt.figure(figsize=(12, 6))
    plt.hist(df_current["watch_time"], color="#4285f4", bins=30, alpha=0.5, density=True, edgecolor="black", label="Current (Recent)")
    plt.hist(df_base["watch_time"], color="#00ab75", bins=30, alpha=0.5, density=True, edgecolor="black", label="Base (Historical)")

    plt.title("Data Drift Visualization: Watch Time Distribution Change")
    plt.xlabel("Watch Time (seconds)")
    plt.ylabel("Density")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

    print("부연 설명: 초록색(Base) 분포에 비해 파란색(Current) 분포의 중심값이 우측으로 이동했습니다.")
    print("이는 평균 시청 시간이 늘어난 '데이터 드리프트' 현상을 보여주며, 기존 모델의 예측력이 떨어질 수 있음을 시사합니다.")
else:
    print("df_current 데이터가 없습니다. 상단의 BigQuery 쿼리 셀을 실행해 주세요.")

In [ ]:
import base64
from facets_overview.generic_feature_statistics_generator import GenericFeatureStatisticsGenerator
from IPython.core.display import display, HTML

gfsg = GenericFeatureStatisticsGenerator()
proto = gfsg.ProtoFromDataFrames([
    {"name": "current", "table": df_current},
    {"name": "base", "table": df_base},
])
protostr = base64.b64encode(proto.SerializeToString()).decode("utf-8")

HTML_TEMPLATE = """
        <script src="https://cdnjs.cloudflare.com/ajax/libs/webcomponentsjs/1.3.3/webcomponents-lite.js"></script>
        <link rel="import" href="https://raw.githubusercontent.com/PAIR-code/facets/1.0.0/facets-dist/facets-jupyter.html" >
        <facets-overview id="elem"></facets-overview>
        <script>
          document.querySelector("#elem").protoInput = "{protostr}";
        </script>"""
html = HTML_TEMPLATE.format(protostr=protostr)
display(HTML(html))

In [ ]:
!pip install evidently

In [ ]:
import numpy as np
from evidently.test_suite import TestSuite
from evidently.test_preset import DataStabilityTestPreset
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

In [ ]:
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset

    if 'df_current' in locals() and not df_current.empty:
        data_drift_report = Report(metrics=[DataDriftPreset()])
        data_drift_report.run(current_data=df_current, reference_data=df_base, column_mapping=None)
        data_drift_report.show(mode='inline')
    else:
        print("분석할 데이터프레임이 준비되지 않았습니다. 상단 쿼리 셀을 실행해주세요.")
except ImportError:
    print("evidently 라이브러리가 로드되지 않았습니다. 상단의 !pip install evidently 셀 실행 후 런타임 재시작이 필요할 수 있습니다.")

In [ ]:
data_drift_report = Report(metrics=[
    DataDriftPreset(),
])
data_drift_report.run(current_data=df.iloc[:2000], reference_data=df.iloc[2000:4000], column_mapping=None)
data_drift_report

In [ ]:
# Evidently 라이브러리 강제 재설치 및 캐시 업데이트
!pip install evidently -U -q

import sys
import pandas as pd

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset

    # Evidently Report 생성 및 실행
    drift_report = Report(metrics=[DataDriftPreset()])

    # 상단에서 정의된 df_base, df_current가 있는지 확인
    if 'df_base' in locals() and 'df_current' in locals():
        drift_report.run(reference_data=df_base, current_data=df_current)
        drift_report.show(mode='inline')
    else:
        print("데이터(df_base, df_current)가 존재하지 않습니다. 상단의 BigQuery 쿼리 셀들을 실행해 주세요.")

except (ImportError, ModuleNotFoundError):
    print("라이브러리를 새로 설치했습니다. 효과적인 적용을 위해 '런타임 > 런타임 다시 시작' 후 이 셀을 다시 실행해 주세요.")

### 💡 Evidently AI 리포트 해석 가이드

Evidently AI의 Data Drift 리포트에서 결과를 해석할 때 주목해야 할 핵심 기준들입니다.

#### 1. 주요 지표 (Key Metrics)
*   **p-value**: 통계적 검정 결과입니다. 기본 임계값은 **0.05**입니다.
    *   **p-value < 0.05**: 데이터 분포의 변화가 통계적으로 유의미함 (Drift 발생).
    *   **p-value ≥ 0.05**: 분포의 변화가 우연일 가능성이 높음 (No Drift).
*   **Drift Share**: 전체 특징(Feature) 중 드리프트가 감지된 컬럼의 비율입니다.
    *   기본적으로 **50% 이상의 컬럼**에서 드리프트가 발생하면 데이터셋 전체에 드리프트가 있다고 판단합니다.

#### 2. 데이터 타입별 기본 통계 검정
Evidently는 데이터의 양과 타입에 따라 자동으로 최적의 알고리즘을 선택합니다.
*   **수치형 데이터 (Numerical)**:
    *   데이터가 1,000개 미만: **K-S Test (Kolmogorov-Smirnov)**
    *   데이터가 1,000개 이상: **Wasserstein Distance**
*   **범주형 데이터 (Categorical)**:
    *   데이터가 1,000개 미만: **Chi-squared Test**
    *   데이터가 1,000개 이상: **Jensen-Shannon Distance**

#### 3. 리포트의 시각적 요소
*   **Distributions**: Reference(과거)와 Current(현재)의 히스토그램을 겹쳐서 보여줍니다. 두 그래프의 겹치는 면적이 작을수록 드리프트가 심한 것입니다.
*   **Correlations**: 특징 간의 상관관계 변화를 보여주며, 특정 변수 간의 관계가 깨졌을 때 모델 성능 하락을 예측할 수 있습니다.

In [ ]:
import pandas as pd
import numpy as np

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset

    # 1. drift_report가 없는 경우를 대비한 안전 장치 (데모용 데이터 생성)
    if 'drift_report' not in locals():
        print("참조할 drift_report가 없어 샘플 데이터를 생성하여 실행합니다.")
        data_ref = pd.DataFrame({'feature': np.random.randn(100)})
        data_cur = pd.DataFrame({'feature': np.random.randn(100) + 2}) # 드리프트 발생 유도
        drift_report = Report(metrics=[DataDriftPreset()])
        drift_report.run(reference_data=data_ref, current_data=data_cur)

    # 2. 리포트 결과를 JSON(Dict) 형식으로 변환
    report_json = drift_report.as_dict()

    # 3. 핵심 지표 추출 (Dataset Drift Metric 기준)
    drift_share = report_json['metrics'][0]['result']['drift_share']
    dataset_drift = report_json['metrics'][0]['result']['dataset_drift']

    # 4. 알람 및 재학습 트리거 조건 설정
    DRIFT_THRESHOLD = 0.5

    print(f"현재 드리프트된 피처 비율: {drift_share:.2%}")

    if dataset_drift or drift_share >= DRIFT_THRESHOLD:
        print("🚨 [ALARM] 데이터 드리프트 임계값 초과!")
        print(">>> 조치 사항: 모델 성능 정밀 점검 및 재학습 파이프라인 트리거")
    else:
        print("✅ 데이터 분포가 안정적입니다. 모델 서비스를 지속합니다.")

except ModuleNotFoundError:
    print("❌ 'evidently' 라이브러리를 찾을 수 없습니다.")
    print("상단 메뉴에서 [런타임] > [런타임 다시 시작]을 누른 후 이 셀을 다시 실행해 주세요.")

❌ 'evidently' 라이브러리를 찾을 수 없습니다.
상단 메뉴에서 [런타임] > [런타임 다시 시작]을 누른 후 이 셀을 다시 실행해 주세요.


In [ ]:
import pandas as pd
import numpy as np
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset

    # 1. drift_report가 없는 경우를 대비한 안전 장치
    if 'drift_report' not in locals():
        print("참조할 drift_report가 없어 샘플 데이터를 생성합니다.")
        data_ref = pd.DataFrame({'watch_time': np.random.randn(100) * 100 + 2000})
        data_cur = pd.DataFrame({'watch_time': np.random.randn(100) * 120 + 4000})
        drift_report = Report(metrics=[DataDriftPreset()])
        drift_report.run(reference_data=data_ref, current_data=data_cur)

    # 2. 리포트 결과를 딕셔너리로 변환
    report_dict = drift_report.as_dict()

    # 3. 특정 피처(예: 'watch_time')의 드리프트 상세 정보 추출
    target_feature = 'watch_time'
    drift_metrics = report_dict['metrics'][0]['result']['drift_by_feature']

    if target_feature in drift_metrics:
        feature_data = drift_metrics[target_feature]

        print(f"--- [{target_feature}] 드리프트 상세 분석 ---")
        print(f"- 드리프트 여부: {feature_data['drift_detected']}")
        print(f"- p-value (또는 score): {feature_data['drift_score']:.4f}")
        print(f"- 통계 검정법: {feature_data['stat_test_name']}")
        print(f"- 임계값(Threshold): {feature_data['threshold']}")
    else:
        print(f"'{target_feature}' 피처를 리포트에서 찾을 수 없습니다.")
        print(f"분석 가능한 피처 목록: {list(drift_metrics.keys())}")

except ModuleNotFoundError:
    print("❌ 'evidently' 라이브러리가 설치되지 않았습니다. 상단에서 설치 및 런타임 재시작을 먼저 진행해 주세요.")

❌ 'evidently' 라이브러리가 설치되지 않았습니다. 상단에서 설치 및 런타임 재시작을 먼저 진행해 주세요.


In [ ]:
try:
    from evidently.calculations.stattests import stattest_registry

    # 1. 사용 가능한 모든 통계 검정법(StatTests) 목록 가져오기
    all_stattests = stattest_registry.list_stattests()

    print(f"--- Evidently AI 지원 통계 검정법 (총 {len(all_stattests)}개) ---\n")
    for test_name in sorted(all_stattests):
        print(f"- {test_name}")

    # 2. 특정 상황에서 Evidently가 권장하는 기본 검정법 설명
    print("\n💡 참고: Evidently는 데이터 타입과 샘플 수에 따라 최적의 검정법을 자동 선택합니다.")
    print("- 'ks': Kolmogorov-Smirnov (수치형, 소규모 데이터)")
    print("- 'psi': Population Stability Index (분포 안정성)")
    print("- 'chisquare': Chi-squared (범주형, 소규모 데이터)")
    print("- 'jensenshannon': Jensen-Shannon distance (범주형, 대규모 데이터)")
    print("- 'wasserstein': Wasserstein distance (수치형, 대규모 데이터)")

except ModuleNotFoundError:
    print("❌ 'evidently' 라이브러리가 로드되지 않았습니다. [런타임 다시 시작] 후 확인해 주세요.")

❌ 'evidently' 라이브러리가 로드되지 않았습니다. [런타임 다시 시작] 후 확인해 주세요.


In [ ]:
### 6. 결론 및 향후 조치

실습 결과, 다음과 같은 관찰 사항과 후속 조치를 도출할 수 있습니다.

1.  **통계적 유의성**: 시각화와 Evidently 리포트를 통해 `watch_time` 분포의 뚜렷한 변화(Drift)를 확인했습니다.
2.  **모니터링 알람**: 실제 운영 환경에서는 이러한 변화가 감지될 경우 PSI(Population Stability Index) 등의 지표를 기준으로 자동 알람을 설정해야 합니다.
3.  **모델 재학습**: 데이터의 특성이 변했으므로, 현재의 분포를 반영할 수 있도록 최신 데이터를 포함하여 모델을 재학습(Retraining)하는 파이프라인 트리거가 필요합니다.

### ## Resources

모델 드리프트 감지에 활용할 수 있는 파이썬 라이브러리들을 추천해 드릴게요:

*   **Evidently AI**: 이 라이브러리는 이미 노트북에서도 사용하고 계시지만, 데이터 드리프트, 모델 성능 드리프트, 데이터 품질 등에 대한 상세한 리포트를 시각화하여 제공하는 데 매우 강력합니다. 다양한 통계 테스트와 시각화 기능을 포함하고 있습니다.
*   **NannyML**: 모델 성능 하락의 주요 원인 중 하나인 데이터 드리프트를 조기에 감지하는 데 특화된 라이브러리입니다. 실제 모델 성능에 미치는 영향을 추정하면서 드리프트를 감지하는 기능을 제공하여 '개념 드리프트'와 '데이터 드리프트'를 모두 다룰 수 있도록 돕습니다.
*   **Alibi Detect**: 이 라이브러리는 아웃라이어, 드리프트, 적대적 공격 감지 등 머신러닝 모델의 신뢰성과 관련한 다양한 기능을 제공합니다. 특히, 데이터 드리프트 감지를 위한 여러 통계적 방법론(예: KS-test, Chi-squared test, Maximum Mean Discrepancy (MMD))을 구현하고 있어 유연하게 사용할 수 있습니다.
*   **Deepchecks**: 데이터 유효성 검사, 모델 성능 평가, 데이터 드리프트 감지 등 ML 모델 라이프사이클 전반에 걸쳐 유용한 툴킷을 제공합니다. 사용하기 쉬운 인터페이스와 자세한 리포트가 특징입니다.
*   **Great Expectations**: 데이터 품질과 유효성 검증에 중점을 둔 라이브러리입니다. 데이터가 특정 '기대치(Expectations)'를 충족하는지 검증하여 데이터 드리프트를 간접적으로 감지할 수 있습니다. 데이터 파이프라인의 데이터 품질을 유지하는 데 매우 유용합니다.

### 💡 실무 적용 시 고려 사항

학습한 라이브러리들을 실제 프로젝트나 운영 환경에 적용할 때 고려해야 할 핵심 사항 5가지를 정리해 드립니다:

1.  **감지 시점과 자동화 (Monitoring Pipeline)**: 단순히 코드를 실행하는 것에 그치지 않고, CI/CD 파이프라인이나 MLOps 워크플로우(예: Kubeflow, Airflow)에 통합하여 주기적으로 드리프트를 체크하고 관리자에게 알람을 보내는 자동화가 필요합니다.
2.  **임계값(Threshold) 설정**: 통계적 유의성(p-value)이 있다고 해서 반드시 모델 성능에 치명적인 것은 아닙니다. 비즈니스 영향도를 고려하여 '재학습이 필요한 시점'을 정의하는 적절한 PSI 또는 드리프트 스코어 임계값을 실험을 통해 설정해야 합니다.
3.  **데이터 샘플링 전략**: 대규모 데이터셋의 경우 모든 데이터를 비교하는 것은 리소스 소모가 큽니다. 분석에 충분한 통계적 검정력을 유지하면서도 계산 효율성을 높일 수 있는 적절한 샘플링 전략이 필요합니다.
4.  **참조 데이터(Reference Data)의 관리**: 드리프트를 판단하는 기준이 되는 'Base Data'를 무엇으로 할지 결정해야 합니다. 초기 학습 데이터로 고정할 것인지, 혹은 최근 1개월 데이터를 롤링(Rolling) 방식으로 사용할지에 따라 감지되는 양상이 달라집니다.
5.  **가짜 알람(False Positives) 대응**: 계절성(Seasonality)이나 일시적인 이벤트로 인해 데이터 분포가 변할 수 있습니다. 무조건적인 재학습보다는 드리프트의 원인이 데이터 품질 문제인지, 실제 시장의 변화인지 분석하는 과정이 선행되어야 합니다.